In [11]:
import pandas as pd
import json
import requests
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

# --- 1. Carregamento e Preparação dos Dados (a partir de uma URL) ---
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

try:
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.RequestException as e:
    print(f"Erro ao buscar dados da URL: {e}")
    exit()

# "Achatamos" a estrutura JSON para uma tabela (DataFrame) do Pandas
df = pd.json_normalize(data, sep='_')

print (df.columns)

# Vamos limpar os dados: Converter SeniorCitizen para algo mais legível e remover colunas numéricas contínuas
df['customer_SeniorCitizen'] = df['customer_SeniorCitizen'].apply(lambda x: 'Yes' if x == 1 else 'No')
df = df.drop(columns=['account_Charges_Monthly', 'account_Charges_Total'])

# --- 2. Limpeza e Transformação dos Dados (AÇÕES CORRETIVAS AQUI) ---

# <- AQUI 1: Remover a coluna de ID, que não serve para encontrar padrões.
df_clean = df.drop(columns=['customerID'])

# <- AQUI 2: Discretizar (binning) a coluna numérica 'tenure'.
# Vamos criar faixas de tempo de permanência do cliente.
bins = [0, 12, 36, 60, 100] # Faixas: 0-12 meses, 13-36, 37-60, >60
labels = ['Cliente Novo (0-1 ano)', 'Cliente Médio (1-3 anos)', 'Cliente Antigo (3-5 anos)', 'Cliente Leal (>5 anos)']
df_clean['tenure_group'] = pd.cut(df_clean['customer_tenure'], bins=bins, labels=labels, right=False)

# Agora podemos remover a coluna numérica original 'tenure'
df_clean = df_clean.drop(columns=['customer_tenure'])

# Converter SeniorCitizen para algo mais legível
df_clean['customer_SeniorCitizen'] = df_clean['customer_SeniorCitizen'].apply(lambda x: 'Yes' if x == 1 else 'No')

print("--- DataFrame Limpo e Transformado (Pronto para Análise) ---")
print(df_clean.head())
print("\n" + "="*50 + "\n")


# --- 3. Foco nos Clientes que Desistiram (Churn = "Yes") ---

df_churn = df_clean[df_clean['Churn'] == 'Yes'].copy()
df_churn = df_churn.drop(columns=['Churn'])

print(f"--- Análise focada em {len(df_churn)} clientes que desistiram (Churn='Yes') ---")
print(df_churn.head())
print("\n" + "="*50 + "\n")


# --- 4. Transformação dos Dados para o Formato do Apriori ---

# Agora o get_dummies funcionará perfeitamente, pois só há colunas categóricas
df_encoded = pd.get_dummies(df_churn)

print("--- DataFrame Codificado para o Apriori (formato one-hot) ---")
print(df_encoded.head())
print("\n" + "="*50 + "\n")


# --- AJUSTE DE VISUALIZAÇÃO COMPLETO PARA O PANDAS ---
# Dizemos ao pandas para não esconder NADA ao imprimir.

# Mostra todas as linhas (sem truncar com '...')
pd.set_option('display.max_rows', None)

# Mostra todas as colunas (sem truncar com '...')
pd.set_option('display.max_columns', None)

# Aumenta a largura total da linha de exibição para evitar quebra de linha
pd.set_option('display.width', 2000)

# Aumenta a largura do conteúdo de uma única coluna para que os 'itemsets' não sejam truncados
pd.set_option('display.max_colwidth', None)

# --- 5. Aplicação do Algoritmo Apriori ---

min_support_threshold = 0.3
frequent_itemsets = apriori(df_encoded, min_support=min_support_threshold, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))


# --- 6. ANÁLISE DOS PERFIS DE CLIENTES (Frequent Itemsets) ---

print("="*60)
print("🎯 PERFIS DE MAIOR RISCO (Combinações mais comuns em clientes que cancelaram)")
print("="*60)
# Focamos em perfis com pelo menos 3 características para insights mais profundos
# e ordenamos pelo suporte (mais comum primeiro)
top_profiles = frequent_itemsets[frequent_itemsets['length'] >= 3].sort_values(by='support', ascending=False)
print(top_profiles.head(15)) # Mostramos os 15 perfis mais relevantes
print("\n" * 2)


# --- 7. ANÁLISE DOS GATILHOS DE CANCELAMENTO (Association Rules) ---

# Geramos as regras a partir dos conjuntos frequentes
# Usaremos um limiar de confiança, mas o 'lift' será nosso principal guia
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print("="*60)
print("🚀 GATILHOS DE CANCELAMENTO (Regras de associação com maior poder preditivo)")
print("="*60)
print("Analisando regras onde a associação NÃO é coincidência (lift > 1.2) e a confiança é alta.")

# Filtramos para encontrar as regras MAIS INTERESSANTES:
# - lift > 1.2: A regra é pelo menos 20% mais provável que o acaso. Isso remove o ruído.
# - confidence > 0.8: A regra é muito confiável (80% de chance).
strong_rules = rules[(rules['lift'] > 1.2) & (rules['confidence'] > 0.8)].sort_values(by=['lift', 'confidence'], ascending=[False, False])

print(strong_rules.head(15)) # Mostramos as 15 regras mais fortes
print("\n")

Index(['customerID', 'Churn', 'customer_gender', 'customer_SeniorCitizen', 'customer_Partner', 'customer_Dependents', 'customer_tenure', 'phone_PhoneService', 'phone_MultipleLines', 'internet_InternetService', 'internet_OnlineSecurity', 'internet_OnlineBackup', 'internet_DeviceProtection', 'internet_TechSupport', 'internet_StreamingTV', 'internet_StreamingMovies', 'account_Contract', 'account_PaperlessBilling', 'account_PaymentMethod', 'account_Charges_Monthly', 'account_Charges_Total'], dtype='object')
Index(['customerID', 'Churn', 'customer_gender', 'customer_SeniorCitizen', 'customer_Partner', 'customer_Dependents', 'customer_tenure', 'phone_PhoneService', 'phone_MultipleLines', 'internet_InternetService', 'internet_OnlineSecurity', 'internet_OnlineBackup', 'internet_DeviceProtection', 'internet_TechSupport', 'internet_StreamingTV', 'internet_StreamingMovies', 'account_Contract', 'account_PaperlessBilling', 'account_PaymentMethod'], dtype='object')
--- DataFrame Limpo e Transformado

/usr/local/lib/python3.11/dist-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


### Análisis Detallado de los Resultados Finales

#### 1. El Perfil Indiscutible del Cliente que Cancela (Tabla `🎯 PERFILES DE MAYOR RIESGO`)

Esta tabla nos muestra el "ADN" del cliente propenso a la cancelación. Los resultados son extremadamente claros.

*   **El 80.4% de todos los clientes que cancelan son `(No es Adulto Mayor + Tiene Servicio Telefónico + Contrato Mensual)` (Línea 288).**
    *   La falta de compromiso a largo plazo (`Month-to-month`) es el factor más crítico.

*   **El Refuerzo del Perfil "Desapegado" (Líneas 261 y 273): ~75% de los clientes que cancelan tampoco `no tienen dependientes`.**
    *   Al unirlo con el punto anterior, el perfil se consolida: `(No es Adulto Mayor + Sin Dependientes + Contrato Mensual)`. Estamos hablando de individuos o parejas sin hijos, que poseen una barrera de salida muy baja.

*   **El Rechazo de los Servicios de Valor Agregado (Líneas 346, 323, 280, 283):**
    *   **72.2%** no tienen `Soporte Técnico`.
    *   **71.9%** no tienen `Seguridad Online`.
    *   Esto no es coincidencia. Estos clientes activamente **no quieren involucrarse** con el ecosistema de la empresa. Buscan el servicio básico y nada más, tratando a la empresa como un proveedor de un commodity, no como un socio.

*   **El Factor de la Fibra Óptica (Línea 279): 69.4% de los que cancelan son `(No es Adulto Mayor + Tiene Servicio Telefónico + Contratan Fibra Óptica)`.**
    *   Este es un hallazgo crucial. El problema de la cancelación no está en los servicios de baja calidad, ¡sino en nuestro producto más premium! Los clientes contratan la mejor internet, pero no encuentran motivos para permanecer. Esto puede indicar que el precio de la Fibra en el plan mensual se percibe como muy alto sin los beneficios de un contrato a largo plazo, o que la competencia en este segmento es feroz.

**Conclusión de los Perfiles:**
El cliente propenso a la cancelación es un **adulto (no mayor), sin dependientes, que opta por la máxima flexibilidad del contrato mensual y ve poco o ningún valor en nuestros servicios de soporte y seguridad**. Es un "consumidor transaccional" que, incluso pudiendo contratar nuestro mejor servicio (Fibra), no establece ningún lazo de lealtad con la marca.

---

#### 2. La Reacción en Cadena hacia la Cancelación (Tabla `🚀 GATILLOS DE CANCELACIÓN`)

Esta tabla revela la "psicología" detrás de la decisión de irse. Las reglas son complejas, pero la historia que cuentan es simple y poderosa.

*   **La Regla Más Fuerte de Todas (Línea 27718, Lift = 1.52):**
    *   **SI** un cliente es `soltero/sin pareja`, tiene `Fibra Óptica` y `no tiene Soporte Técnico`...
    *   **ENTONCES** hay un **84% de certeza** de que se ajusta a todo el perfil de cancelación: `(no tiene Seguridad Online + tiene Teléfono + no tiene Dependientes + está en Contrato Mensual)`.
    *   **Significado:** El rechazo de un solo servicio de valor (`TechSupport`) en un cliente con perfil "individualista" (`Partner_No`) es una señal de alerta máxima. Predice con altísima confianza todo el resto del comportamiento de bajo involucramiento que conduce a la cancelación.

*   **El Contrato Mensual como Consecuencia Inevitable:**
    *   Observe cómo `account_Contract_Month-to-month` aparece repetidamente en los **consecuentes** (la parte "ENTONCES" de la regla).
    *   **Significado:** Otros comportamientos (no tener pareja, no tener seguridad, tener fibra) son los **detonantes** que llevan al cliente a tomar la decisión final de quedarse en un contrato mensual, que es la puerta de salida. La falta de compromiso contractual no es la causa raíz, sino el **síntoma final** de una serie de decisiones de bajo involucramiento.

**Conclusión de los Detonantes:**
La cancelación de clientes sigue un patrón de comportamiento claro. Comienza con un perfil demográfico (individuos sin lazos familiares fuertes), se manifiesta en el rechazo explícito de servicios que crean dependencia y valor (seguridad, soporte), y culmina en la elección racional de un contrato sin ataduras (mensual), convirtiendo la cancelación en una decisión fácil y de baja fricción.

---

### Diagnóstico Final y Recomendaciones Estratégicas (Su Conclusión Escrita)

**Título:** **Análisis de Cancelación de Clientes: La Paradoja del "Consumidor Premium Desapegado"**

**Diagnóstico:**
El análisis revela que la causa principal de la cancelación de clientes no es la insatisfacción con la calidad del servicio, sino una falla estratégica en crear lazos de lealtad con un segmento específico y dominante: el **"consumidor individualista"**. Este perfil, que representa más del 80% de las cancelaciones, está compuesto por adultos (no mayores) y sin dependientes, que, a pesar de contratar a menudo el servicio más premium (Fibra Óptica), tratan a la empresa de forma puramente transaccional.

La cancelación ocurre por una reacción en cadena:
1.  **Perfil de Bajo Vínculo:** Clientes sin pareja o dependientes son naturalmente más propensos al cambio.
2.  **Rechazo de "Anclas":** Rechazan activamente servicios de valor agregado (Soporte Técnico, Seguridad Online), que funcionan como "anclas" de lealtad, señalando una falta de interés en profundizar en nuestro ecosistema.
3.  **Opción por la Salida Fácil:** Esta mentalidad conduce directamente a la elección del contrato mensual, que elimina cualquier barrera para la cancelación cuando surge una oferta competidora.

**Recomendaciones Accionables:**

1.  **Campaña "Proyecto Ancla":**
    *   **Público Objetivo:** Segmentar proactivamente a clientes con el perfil `(No Adulto Mayor + Sin Pareja/Dependientes + Contrato Mensual + Fibra Óptica)`.
    *   **Acción:** Ofrecer un paquete irresistible para migrar al contrato anual, incluyendo 12 meses gratuitos de `OnlineSecurity` y `TechSupport`. El objetivo es **crear artificialmente las anclas** que ellos no eligen orgánicamente.

2.  **Optimización del Proceso de Contratación:**
    *   **Acción:** Modificar el flujo de contratación para presentar el plan anual como la **"Opción Inteligente"** por defecto, destacando el ahorro y los beneficios incluidos. El plan mensual debe posicionarse como una "Opción Flexible", pero más cara y con menos ventajas.

3.  **Innovación en Paquetes Individuales:**
    *   **Acción:** Desarrollar y probar nuevos paquetes enfocados en el perfil individualista que valora el rendimiento. Ejemplos: "Plan Gamer" (con latencia optimizada y un servicio de VPN), "Plan Streamer" (con velocidad de subida potenciada y suscripciones a servicios de streaming asociados). El objetivo es reemplazar el valor de los servicios que rechazan por beneficios que sí desean, justificando un compromiso a largo plazo.